In [11]:
import json
import numpy as np
import pandas as pd
import time

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split

In [8]:
with open('data\sentences.json', 'r') as file:
    data = json.load(file)

In [9]:
data

[{'text': "Chinese-owned companies are aggressively expanding cobalt mining in Congo and Indonesia even while prices crash, as they bid to raise market share of the metal used in batteries for the country's electric vehicle (EV) industry.",
  'label': 0},
 {'text': "Chinese cobalt producers have seemed unfazed by oversupply that has knocked prices down by more than half in the past 18 months, with some said to benefit from state support for a sector seen as vital to China's EV industry, as well as firmer prices of metals with which cobalt is mined, such as copper.",
  'label': 0},
 {'text': "China's CMOC Group, which boosted its cobalt output by 144% during the first three quarters of 2023, is now on track to become the world's biggest cobalt producer, overtaking commodity group Glencore.",
  'label': 0},
 {'text': 'CMOC is due to lift its market share of the global mined cobalt market from 11% in 2022 to nearly 30% by 2025, said Jorge Uzcategui, an analyst at consultancy Benchmark Min

In [10]:
corpus = []

for sentence in data:
    corpus.extend(sentence['text'].split(' '))           # generate the corpus by extracting text from each sentence, split the list on spaces and adding it to corpus

In [11]:
len(corpus)

648529

tokenize the corpus

In [12]:
tokenizer = Tokenizer(lower=False, filters='')            # this will retain the word structure like chinese-owned and will not remove hyphen
tokenizer.fit_on_texts(corpus)

In [13]:
len(tokenizer.word_index)

49346

In [15]:
# save tokenizer

tokenizer_json = tokenizer.to_json()
with open("artifacts/tokenizer_v1.json", 'w') as f:
    f.write(json.dumps(tokenizer_json))

creating input and output combination. [1st word, second word], [1st word, 2nd word, 3rd word] and so on. last element of each list will be the output and remaining prior to last will be input

In [8]:
sentence_list = [item['text'] for item in data]

In [9]:
len(sentence_list)

25032

In [10]:
input_sequences = []

for sentence in sentence_list:
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i+1])


In [11]:
len(input_sequences)

623489

padding the input sequences to make them of same length

In [12]:
# max length of the input_sequence
max_len = np.argmax([len(x) for x in input_sequences])

In [13]:
# padding upto 30 elements in each sequence

padded_input_sequences = pad_sequences(input_sequences, maxlen = 30, padding = 'pre')

Extract Input and Output features from padded sequences. last element from each sequence is output and remaining will be input

In [14]:
X = padded_input_sequences[:, :-1]
y = padded_input_sequences[:, -1]

Model Building and compilation

In [48]:
vocab_size = len(tokenizer.word_index) + 1

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=100))
model.add(LSTM(150, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(vocab_size, activation = 'softmax'))


In [49]:
model.compile(loss = 'sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [50]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [51]:
es = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)       # will stop the training in case of overfitting (training loss keeps going down but validation loss start going up)

model.fit(X, y, epochs = 100, validation_split=0.1, verbose = 1, callbacks=[es])

Epoch 1/100
17536/17536 ━━━━━━━━━━━━━━━━━━━━ 5291s 301ms/step - accuracy: 0.1074 - loss: 7.2050 - val_accuracy: 0.1435 - val_loss: 6.8334
Epoch 2/100
17536/17536 ━━━━━━━━━━━━━━━━━━━━ 4511s 257ms/step - accuracy: 0.1708 - loss: 6.1957 - val_accuracy: 0.1699 - val_loss: 6.6009
Epoch 3/100
17536/17536 ━━━━━━━━━━━━━━━━━━━━ 3642s 208ms/step - accuracy: 0.2013 - loss: 5.7285 - val_accuracy: 0.1765 - val_loss: 6.6187
Epoch 4/100
17536/17536 ━━━━━━━━━━━━━━━━━━━━ 3573s 204ms/step - accuracy: 0.2223 - loss: 5.3964 - val_accuracy: 0.1823 - val_loss: 6.6098


# Predictions

In [ ]:
import time

text = 'oil prices hit new'

for i in range(15):
    #tokenize
    token_text = tokenizer.texts_to_sequences([text])[0]

    #padding
    padded_token_text = pad_sequences([token_text], maxlen = 29, padding = 'pre')

    #predict
    import numpy as np
    pos = np.argmax(model.predict(padded_token_text))

    for word, index in tokenizer.word_index.items():
        if index == pos:
            text = text + ' ' + word

            print(text)
            time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step
oil prices hit new lowest
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
oil prices hit new lowest since
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
oil prices hit new lowest since Nov.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
oil prices hit new lowest since Nov. 28,
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
oil prices hit new lowest since Nov. 28, and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
oil prices hit new lowest since Nov. 28, and the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
oil prices hit new lowest since Nov. 28, and the dollar
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
oil prices hit new lowest since Nov. 28, and the dollar was
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
oil prices hit new lowest since Nov. 28, and the dollar was down
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
oil prices hit new lowest since Nov. 28, and the dollar was down 0.4%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
oil prices hit new lowest since Nov. 28, and the dollar was down 0.4% to
1/1 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
model.save('artifacts\model_v1.keras')

In [ ]:
# save tokenizer


In [82]:
text = 'currently'

#tokenize
token_text = tokenizer.texts_to_sequences([text])[0]

#padding
padded_token_text = pad_sequences([token_text], maxlen = 56, padding = 'pre')


top_3_indices  = np.argsort(model.predict([padded_token_text])[0])[-3:][::-1]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step


In [83]:
suggestions = []

for i in top_3_indices:
    for word, index in tokenizer.word_index.items():
        if index == i:
            suggestions.append(word)

print(suggestions)



['also', 'has', 'in']


Loading saved model to try

In [13]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import tokenizer_from_json

with open('artifacts/tokenizer_v1.json') as file:
    tokenizer_data = json.load(file)

tokenizer = tokenizer_from_json(tokenizer_data) 
model1 = load_model('artifacts\model_v1.keras')
model2 = load_model('artifacts\model_v2.keras')

In [14]:
def test_model(text, model):
    for i in range(15):
        #tokenize
        token_text = tokenizer.texts_to_sequences([text])[0]

        #padding
        padded_token_text = pad_sequences([token_text], maxlen = 29, padding = 'pre')

        #predict
        import numpy as np
        pos = np.argmax(model.predict(padded_token_text))

        for word, index in tokenizer.word_index.items():
            if index == pos:
                text = text + ' ' + word

                print(text)
                time.sleep(2)

In [16]:
text = 'Economic growth slowed'
test_model(text, model)



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
Economic growth slowed in
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Economic growth slowed in November
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step
Economic growth slowed in November with
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step
Economic growth slowed in November with the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step
Economic growth slowed in November with the Federal
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
Economic growth slowed in November with the Federal Reserve
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Economic growth slowed in November with the Federal Reserve and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
Economic growth slowed in November with the Federal Reserve and the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Economic growth slowed in November with the Federal Reserve and the Federal
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
Economic growth slowed in November with the Federal Reserve and the Federal Reserve
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Economic growth slowed in 

In [17]:
test_model(text, model2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
Economic growth slowed in
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Economic growth slowed in November
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
Economic growth slowed in November and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
Economic growth slowed in November and the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
Economic growth slowed in November and the unemployment
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
Economic growth slowed in November and the unemployment rate
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Economic growth slowed in November and the unemployment rate fell
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
Economic growth slowed in November and the unemployment rate fell to
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
Economic growth slowed in November and the unemployment rate fell to 3.7%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
Economic growth slowed in November and the unemployment rate fell to 3.7% in
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
Economic growth slowed in N